# PulmoScan AI – Lung Cancer Risk Prediction
## Complete ML Notebook

**Dataset:** Survey Lung Cancer (Kaggle)  
**Goal:** Train and compare 5 ML algorithms, select best model, export artifacts  
**DISCLAIMER:** For educational purposes only. Not a medical diagnosis tool.

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_curve, auc
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
print('✅ All imports successful')

## 2. Load & Inspect Dataset

In [ ]:
# Load real Kaggle lung cancer survey dataset
DATA_PATH = 'data/survey_lung_cancer.csv'
df_raw = pd.read_csv(DATA_PATH)

# Normalize column names
df_raw.columns = [c.strip().upper().replace(' ', '_') for c in df_raw.columns]

print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

In [ ]:
print('Data Types:')
print(df_raw.dtypes)
print('\nClass Distribution:')
print(df_raw['LUNG_CANCER'].value_counts())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df_raw['LUNG_CANCER'].value_counts()
axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#ef4444', '#22c55e'], startangle=90)
axes[0].set_title('Lung Cancer Class Distribution', fontsize=13, fontweight='bold')

# Gender breakdown
gender_counts = df_raw['GENDER'].value_counts()
axes[1].bar(gender_counts.index, gender_counts.values,
            color=['#2578e8', '#ec4899'], edgecolor='white', linewidth=2)
axes[1].set_title('Gender Distribution', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('docs/eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Age distribution by lung cancer outcome
plt.figure(figsize=(10, 5))
for label, color in [('YES', '#ef4444'), ('NO', '#22c55e')]:
    subset = df_raw[df_raw['LUNG_CANCER'] == label]['AGE']
    plt.hist(subset, bins=20, alpha=0.6, label=f'Lung Cancer: {label}', color=color)
plt.xlabel('Age')
plt.ylabel('Count')
plt.title('Age Distribution by Lung Cancer Outcome', fontsize=13, fontweight='bold')
plt.legend()
plt.savefig('docs/age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Missing Value Analysis

In [ ]:
# Missing values
missing = df_raw.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found ✅')

# Visualize if any
plt.figure(figsize=(12, 4))
sns.heatmap(df_raw.isnull(), yticklabels=False, cbar=True, cmap='Blues')
plt.title('Missing Value Heatmap', fontsize=13, fontweight='bold')
plt.savefig('docs/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Data Preprocessing

In [ ]:
df = df_raw.copy()

# Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed: {before - len(df)}')

# Fill missing values with mode
for col in df.columns:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)
        print(f'  Filled: {col}')

# Encode GENDER: M→1, F→0
df['GENDER'] = df['GENDER'].map({'M': 1, 'F': 0, 'MALE': 1, 'FEMALE': 0}).fillna(0).astype(int)

# Encode target
le = LabelEncoder()
df['LUNG_CANCER'] = le.fit_transform(df['LUNG_CANCER'].str.upper().str.strip())
print(f'Classes: {le.classes_}  →  {list(range(len(le.classes_)))}')
print(f'\nFinal shape: {df.shape}')
df.head()

## 6. Outlier Analysis

In [ ]:
# Boxplot for AGE (only continuous numeric feature)
plt.figure(figsize=(8, 4))
plt.boxplot(df['AGE'], vert=False, patch_artist=True,
            boxprops=dict(facecolor='#bfe3fd', color='#2578e8'),
            medianprops=dict(color='#ef4444', linewidth=2))
plt.title('AGE – Outlier Detection (Box Plot)', fontsize=13, fontweight='bold')
plt.xlabel('Age')
plt.savefig('docs/outlier_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

Q1 = df['AGE'].quantile(0.25)
Q3 = df['AGE'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['AGE'] < Q1 - 1.5*IQR) | (df['AGE'] > Q3 + 1.5*IQR)]
print(f'Age outliers (IQR method): {len(outliers)} rows')

## 7. Correlation Heatmap

In [ ]:
plt.figure(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, square=True,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('docs/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Feature Selection & Train/Test Split

In [ ]:
FEATURE_COLS = [
    'AGE', 'GENDER', 'SMOKING', 'YELLOW_FINGERS', 'ANXIETY', 'PEER_PRESSURE',
    'CHRONIC_DISEASE', 'FATIGUE', 'ALLERGY', 'WHEEZING', 'ALCOHOL_CONSUMING',
    'COUGHING', 'SHORTNESS_OF_BREATH', 'SWALLOWING_DIFFICULTY', 'CHEST_PAIN',
]
TARGET_COL = 'LUNG_CANCER'

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Features         : {len(FEATURE_COLS)}')

## 9. Model Training (5 Algorithms)

In [ ]:
MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
    'SVM':                 SVC(probability=True, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in MODELS.items():
    print(f'Training: {name}...')
    model.fit(X_train_sc, y_train)
    y_pred  = model.predict(X_test_sc)
    y_proba = model.predict_proba(X_test_sc)[:, 1]
    cv_acc  = cross_val_score(model, X_train_sc, y_train, cv=cv, scoring='accuracy').mean()

    results[name] = {
        'model':   model,
        'y_pred':  y_pred,
        'y_proba': y_proba,
        'metrics': {
            'accuracy':    round(accuracy_score(y_test, y_pred), 4),
            'precision':   round(precision_score(y_test, y_pred, zero_division=0), 4),
            'recall':      round(recall_score(y_test, y_pred, zero_division=0), 4),
            'f1':          round(f1_score(y_test, y_pred, zero_division=0), 4),
            'roc_auc':     round(roc_auc_score(y_test, y_proba), 4),
            'cv_accuracy': round(cv_acc, 4),
        }
    }
    m = results[name]['metrics']
    print(f'  Accuracy: {m["accuracy"]} | F1: {m["f1"]} | ROC-AUC: {m["roc_auc"]} | CV-Acc: {m["cv_accuracy"]}')

print('\n✅ All models trained')

## 10. Model Evaluation & Comparison

In [ ]:
# Metrics comparison dataframe
metrics_df = pd.DataFrame(
    {name: v['metrics'] for name, v in results.items()}
).T.drop(columns=['cv_accuracy'])

metrics_df.sort_values('roc_auc', ascending=False).style.background_gradient(
    cmap='Blues', axis=0
).format('{:.4f}')

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(12, 6))
metric_keys = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
x = np.arange(len(metric_keys))
width = 0.15
colors = ['#2578e8', '#14b8a6', '#f59e0b', '#ef4444', '#8b5cf6']

for i, (name, data) in enumerate(results.items()):
    vals = [data['metrics'][k] for k in metric_keys]
    ax.bar(x + i*width, vals, width, label=name, color=colors[i], alpha=0.85, edgecolor='white')

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Model Comparison – All Metrics', fontsize=13, fontweight='bold')
ax.set_xticks(x + width*2)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])
ax.set_ylim([0.7, 1.05])
ax.legend()
ax.yaxis.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('docs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC curves for all models
plt.figure(figsize=(10, 7))
for (name, data), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, data['y_proba'])
    auc_val = data['metrics']['roc_auc']
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=color, linewidth=2)

plt.plot([0,1],[0,1], 'k--', alpha=0.5, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves – All Models', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.savefig('docs/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Best Model Selection & Confusion Matrix

In [ ]:
# Select best model by ROC-AUC
best_name = max(results, key=lambda n: results[n]['metrics']['roc_auc'])
best = results[best_name]
print(f'🏆 Best Model: {best_name}')
print(f"   ROC-AUC : {best['metrics']['roc_auc']}")
print(f"   Accuracy: {best['metrics']['accuracy']}")
print(f"   F1-Score: {best['metrics']['f1']}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, best['y_pred'])
fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['NO Cancer', 'YES Cancer'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix – {best_name}', fontsize=13, fontweight='bold')
plt.savefig('docs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, best['y_pred'], target_names=['NO', 'YES']))

## 12. Feature Importance

In [ ]:
# Feature importance (works for tree-based models)
model_obj = best['model']
if hasattr(model_obj, 'feature_importances_'):
    importances = model_obj.feature_importances_
    fi_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': importances})
    fi_df = fi_df.sort_values('importance', ascending=True)

    plt.figure(figsize=(10, 7))
    bars = plt.barh(fi_df['feature'], fi_df['importance'],
                    color=plt.cm.Blues(fi_df['importance'] / fi_df['importance'].max()))
    plt.xlabel('Importance Score')
    plt.title(f'Feature Importance – {best_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('docs/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'{best_name} does not expose feature_importances_. Using permutation importance.')
    from sklearn.inspection import permutation_importance
    perm = permutation_importance(model_obj, X_test_sc, y_test, n_repeats=10, random_state=42)
    fi_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': perm.importances_mean})
    fi_df = fi_df.sort_values('importance', ascending=True)
    fi_df.tail(10).plot.barh(x='feature', y='importance', color='#2578e8', legend=False,
                              title=f'Permutation Importance – {best_name}', figsize=(10,6))
    plt.tight_layout()
    plt.savefig('docs/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

## 13. Save Artifacts

In [ ]:
OUTPUT_DIR = Path('ml_models')
OUTPUT_DIR.mkdir(exist_ok=True)

joblib.dump(model_obj,    OUTPUT_DIR / 'best_model.pkl')
joblib.dump(scaler,       OUTPUT_DIR / 'scaler.pkl')
joblib.dump(le,           OUTPUT_DIR / 'encoder.pkl')
joblib.dump(FEATURE_COLS, OUTPUT_DIR / 'feature_columns.pkl')
(OUTPUT_DIR / 'model_name.txt').write_text(best_name)

# Save metrics JSON
metrics_export = {n: v['metrics'] for n, v in results.items()}
with open(OUTPUT_DIR / 'model_metrics.json', 'w') as f:
    json.dump(metrics_export, f, indent=2)

# Export processed dataset
df.to_csv('data/processed_lung_cancer.csv', index=False)

print('✅ Artifacts saved:')
print('   ml_models/best_model.pkl')
print('   ml_models/scaler.pkl')
print('   ml_models/encoder.pkl')
print('   ml_models/feature_columns.pkl')
print('   ml_models/model_metrics.json')
print('   data/processed_lung_cancer.csv')

---
## Summary

| Model | Accuracy | F1-Score | ROC-AUC |
|---|---|---|---|
| Logistic Regression | — | — | — |
| Random Forest | — | — | — |
| Gradient Boosting | — | — | — |
| SVM | — | — | — |
| Decision Tree | — | — | — |

*(Run notebook to populate)*

**DISCLAIMER:** This notebook is for educational purposes only. Predictions produced by these models should not be used as a substitute for professional medical advice or diagnosis.